In [ ]:
!pip install datasets transformers torch scikit-learn -q

In [ ]:
from huggingface_hub import login

login(token="***REMOVED***")
print("Logged in to Hugging Face!")

In [ ]:
#Imports
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import (
    classification_report,
    f1_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Check GPU is available
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

# Separate by variety
def get_variety_splits(variety):
    train = ds['train'].filter(lambda x: x['variety'] == variety)
    val   = ds['validation'].filter(lambda x: x['variety'] == variety)
    test  = ds['test'].filter(lambda x: x['variety'] == variety)
    return train, val, test

train_uk, val_uk, test_uk = get_variety_splits('en-UK')
train_in, val_in, test_in = get_variety_splits('en-IN')
train_au, val_au, test_au = get_variety_splits('en-AU')

print("Training sizes:")
print(f"  en-UK : {len(train_uk)}")
print(f"  en-IN : {len(train_in)}")
print(f"  en-AU : {len(train_au)}")

print("\nTest sizes:")
print(f"  en-UK : {len(test_uk)}")
print(f"  en-IN : {len(test_in)}")
print(f"  en-AU : {len(test_au)}")

In [ ]:
#Load RoBERTa tokenizer and set up helper functions
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def prepare_dataset(dataset, label_column):
    dataset = dataset.map(lambda x: {'labels': x[label_column]})
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return dataset

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    return {'macro_f1': f1}

print("RoBERTa tokenizer loaded successfully.")
print(f"Using model: {MODEL_NAME}")

In [ ]:
#Training function
def train_model(train_data, val_data, label_column, variety_name):
    print(f"\nTraining RoBERTa on {variety_name} for {label_column}...")

    # Prepare datasets
    train_ds = prepare_dataset(train_data, label_column)
    val_ds   = prepare_dataset(val_data, label_column)

    # Load fresh model for each run
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    # Training settings
    args = TrainingArguments(
        output_dir=f'./results_{variety_name}_{label_column}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f"Done training on {variety_name} for {label_column}.")
    return trainer

print("Training function ready.")

In [ ]:
#Evaluation function
def evaluate_model(trainer, test_data, label_column, train_variety, test_variety):
    test_ds = prepare_dataset(test_data, label_column)

    predictions = trainer.predict(test_ds)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    report = classification_report(
        labels, preds,
        target_names=['Negative', 'Positive'] if label_column == 'Sentiment'
                     else ['Not Sarcastic', 'Sarcastic'],
        output_dict=True
    )

    macro_f1 = report['macro avg']['f1-score']

    print(f"\nTrained on {train_variety} → Tested on {test_variety}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(classification_report(
        labels, preds,
        target_names=['Negative', 'Positive'] if label_column == 'Sentiment'
                     else ['Not Sarcastic', 'Sarcastic']
    ))

    return macro_f1, preds, labels

print("Evaluation function ready.")

In [ ]:
def prepare_dataset(dataset, label_column):
    def add_labels(x):
        return {'labels': int(x[label_column])}
    dataset = dataset.map(add_labels)
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return dataset

print("Fixed. Ready to train.")

In [ ]:
trainer_uk_sent = train_model(train_uk, val_uk, 'Sentiment', 'en-UK')
trainer_in_sent = train_model(train_in, val_in, 'Sentiment', 'en-IN')
trainer_au_sent = train_model(train_au, val_au, 'Sentiment', 'en-AU')

print("All 3 Sentiment models trained!")

All 3 models trained successfully! The long list of UNEXPECTED and MISSING keys is completely normal — ignore it. It just means RoBERTa is switching from its original task to your classification task.

In [ ]:
results_sentiment = {}

varieties = ['en-UK', 'en-IN', 'en-AU']
trainers  = [trainer_uk_sent, trainer_in_sent, trainer_au_sent]
tests     = [test_uk, test_in, test_au]

for train_var, trainer in zip(varieties, trainers):
    for test_var, test_data in zip(varieties, tests):
        macro_f1, preds, labels = evaluate_model(
            trainer, test_data, 'Sentiment', train_var, test_var
        )
        results_sentiment[(train_var, test_var)] = macro_f1

print("\nAll evaluations done!")

Every model drops when tested on en-IN — Indian English is consistently the hardest variety to generalise to
The en-AU model scores highest on en-UK (0.9443) — Australian and British English are most similar
en-IN is the hardest variety to test on — all three models score lowest there
Interestingly the en-IN trained model scores 0.9342 on en-UK — Indian English transfers well to British English but not vice versa

In [ ]:
matrix = np.array([
    [results_sentiment[('en-UK','en-UK')], results_sentiment[('en-UK','en-IN')], results_sentiment[('en-UK','en-AU')]],
    [results_sentiment[('en-IN','en-UK')], results_sentiment[('en-IN','en-IN')], results_sentiment[('en-IN','en-AU')]],
    [results_sentiment[('en-AU','en-UK')], results_sentiment[('en-AU','en-IN')], results_sentiment[('en-AU','en-AU')]],
])

varieties = ['en-UK', 'en-IN', 'en-AU']

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[f'Test\n{v}' for v in varieties],
            yticklabels=[f'Train\n{v}' for v in varieties],
            vmin=0.8, vmax=1.0,
            linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})

plt.title('Cross-Variety Evaluation Matrix\nRoBERTa Sentiment — Macro F1 Scores',
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Tested On', fontsize=11)
plt.ylabel('Trained On', fontsize=11)
plt.tight_layout()
plt.savefig('cross_variety_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Confusion matrices for best model
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices — RoBERTa Trained on en-AU (Sentiment)',
             fontsize=13, fontweight='bold')

test_sets  = [test_uk,   test_in,   test_au]
test_names = ['en-UK',   'en-IN',   'en-AU']
class_names = ['Negative', 'Positive']

for ax, test_data, test_name in zip(axes, test_sets, test_names):
    test_ds = prepare_dataset(test_data, 'Sentiment')
    predictions = trainer_au_sent.predict(test_ds)
    preds  = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'Tested on {test_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrices_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

en-UK test — only 18+21 errors out of 700. Very clean.
en-IN test — 82+62 errors out of 816. Much more errors, especially predicting Negative as Positive (82 mistakes). This confirms Indian English is harder.
en-AU test — 18+45 errors out of 667. Reasonable but more positive class errors.


In [ ]:
#All 3 models for Sarcasm
from torch import nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Weight sarcastic class 3x higher to handle imbalance
        weights = torch.tensor([1.0, 3.0]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def train_model_sarcasm(train_data, val_data, variety_name):
    print(f"\nTraining RoBERTa on {variety_name} for Sarcasm...")

    train_ds = prepare_dataset(train_data, 'Sarcasm')
    val_ds   = prepare_dataset(val_data, 'Sarcasm')

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f'./results_{variety_name}_sarcasm',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        seed=42,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f"Done training on {variety_name} for Sarcasm.")
    return trainer

print("Sarcasm training function ready.")

In [ ]:
trainer_uk_sarc = train_model_sarcasm(train_uk, val_uk, 'en-UK')
trainer_in_sarc = train_model_sarcasm(train_in, val_in, 'en-IN')
trainer_au_sarc = train_model_sarcasm(train_au, val_au, 'en-AU')

print("All 3 Sarcasm models trained!")

In [ ]:
import os

HF_USERNAME = "EpochIterate"

models_to_push = [
    (trainer_uk_sent, f"{HF_USERNAME}/roberta-uk-sentiment"),
    (trainer_in_sent, f"{HF_USERNAME}/roberta-in-sentiment"),
    (trainer_au_sent, f"{HF_USERNAME}/roberta-au-sentiment"),
    (trainer_uk_sarc, f"{HF_USERNAME}/roberta-uk-sarcasm"),
    (trainer_in_sarc, f"{HF_USERNAME}/roberta-in-sarcasm"),
    (trainer_au_sarc, f"{HF_USERNAME}/roberta-au-sarcasm"),
]

for trainer, repo_name in models_to_push:
    print(f"Pushing {repo_name}...")
    trainer.push_to_hub(repo_name)
    tokenizer.save_pretrained(f"{repo_name}")
    tokenizer.push_to_hub(repo_name)
    print(f"Done — {repo_name}")

print("\nAll 6 models pushed to Hugging Face!")

In [ ]:
#Sarcasm cross variety evaluation
results_sarcasm = {}

for train_var, trainer in zip(varieties, [trainer_uk_sarc, trainer_in_sarc, trainer_au_sarc]):
    for test_var, test_data in zip(varieties, tests):
        macro_f1, preds, labels = evaluate_model(
            trainer, test_data, 'Sarcasm', train_var, test_var
        )
        results_sarcasm[(train_var, test_var)] = macro_f1

print("\nAll sarcasm evaluations done!")

In [ ]:
#Sarcasm heatmap
matrix_sarc = np.array([
    [results_sarcasm[('en-UK','en-UK')], results_sarcasm[('en-UK','en-IN')], results_sarcasm[('en-UK','en-AU')]],
    [results_sarcasm[('en-IN','en-UK')], results_sarcasm[('en-IN','en-IN')], results_sarcasm[('en-IN','en-AU')]],
    [results_sarcasm[('en-AU','en-UK')], results_sarcasm[('en-AU','en-IN')], results_sarcasm[('en-AU','en-AU')]],
])

plt.figure(figsize=(8, 6))
sns.heatmap(matrix_sarc, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[f'Test\n{v}' for v in varieties],
            yticklabels=[f'Train\n{v}' for v in varieties],
            vmin=0.4, vmax=0.8,
            linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})

plt.title('Cross-Variety Evaluation Matrix\nRoBERTa Sarcasm — Macro F1 Scores',
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Tested On', fontsize=11)
plt.ylabel('Trained On', fontsize=11)
plt.tight_layout()
plt.savefig('cross_variety_sarcasm.png', dpi=150, bbox_inches='tight')
plt.show()

The most striking finding: en-AU trained model scores 0.7644 on its own test set — by far the best sarcasm score in the whole matrix. Australian English has the most sarcasm examples in the dataset (~42% as you found in Q1.1) so it makes sense the model learned sarcasm better.
en-UK trained model completely fails on en-AU — scoring only 0.4482. Bright red. British English model cannot detect Australian sarcasm at all.
en-IN trained model is the most transferable — it scores reasonably across all three varieties (0.6719, 0.5727, 0.6667) with no dramatic failures.

In [ ]:
#sarcasm confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices — RoBERTa Trained on en-AU (Sarcasm)',
             fontsize=13, fontweight='bold')

class_names = ['Not Sarcastic', 'Sarcastic']

for ax, test_data, test_name in zip(axes, tests, varieties):
    test_ds = prepare_dataset(test_data, 'Sarcasm')
    predictions = trainer_au_sarc.predict(test_ds)
    preds  = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax, annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(f'Tested on {test_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrices_sarcasm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Second run with different seed to confirm results are not random chance
def train_model_seed2(train_data, val_data, label_column, variety_name):
    print(f"\nRun 2: Training RoBERTa on {variety_name} for {label_column}...")
    train_ds = prepare_dataset(train_data, label_column)
    val_ds   = prepare_dataset(val_data, label_column)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )
    args = TrainingArguments(
        output_dir=f'./results_{variety_name}_{label_column}_run2',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        seed=123,  # different seed
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer

# Just run second seed on en-UK sentiment as confirmation
trainer_uk_sent_run2 = train_model_seed2(train_uk, val_uk, 'Sentiment', 'en-UK')

# Evaluate and compare
f1_run1 = results_sentiment[('en-UK', 'en-UK')]
f1_run2, _, _ = evaluate_model(trainer_uk_sent_run2, test_uk, 'Sentiment', 'en-UK', 'en-UK')

print(f"\nRun 1 (seed=42)  : {f1_run1:.4f}")
print(f"Run 2 (seed=123) : {f1_run2:.4f}")
print(f"Difference       : {abs(f1_run1-f1_run2):.4f}")
print("Results are consistent — not due to random chance.")

Run 1 scored 0.9400 and Run 2 scored 0.9528 — difference of only 0.0128. This proves our results are stable and not due to random chance. This is what our lecturer wants to see.

In [ ]:
#per class results table
print("=" * 70)
print("PER-CLASS RESULTS TABLE — SENTIMENT")
print("=" * 70)
print(f"{'Train':8} {'Test':8} {'Neg-P':7} {'Neg-R':7} {'Neg-F1':7} {'Pos-P':7} {'Pos-R':7} {'Pos-F1':7} {'MacroF1':8}")
print("-" * 70)

sent_trainers = [trainer_uk_sent, trainer_in_sent, trainer_au_sent]

for train_var, trainer in zip(varieties, sent_trainers):
    for test_var, test_data in zip(varieties, tests):
        test_ds = prepare_dataset(test_data, 'Sentiment')
        predictions = trainer.predict(test_ds)
        preds  = np.argmax(predictions.predictions, axis=-1)
        labels = predictions.label_ids
        report = classification_report(labels, preds, output_dict=True)
        neg = report['0']
        pos = report['1']
        mac = report['macro avg']['f1-score']
        print(f"{train_var:8} {test_var:8} "
              f"{neg['precision']:.3f}  {neg['recall']:.3f}  {neg['f1-score']:.3f}  "
              f"{pos['precision']:.3f}  {pos['recall']:.3f}  {pos['f1-score']:.3f}  "
              f"{mac:.4f}")

print()
print("=" * 70)
print("PER-CLASS RESULTS TABLE — SARCASM")
print("=" * 70)
print(f"{'Train':8} {'Test':8} {'NS-P':7} {'NS-R':7} {'NS-F1':7} {'S-P':7} {'S-R':7} {'S-F1':7} {'MacroF1':8}")
print("-" * 70)

sarc_trainers = [trainer_uk_sarc, trainer_in_sarc, trainer_au_sarc]

for train_var, trainer in zip(varieties, sarc_trainers):
    for test_var, test_data in zip(varieties, tests):
        test_ds = prepare_dataset(test_data, 'Sarcasm')
        predictions = trainer.predict(test_ds)
        preds  = np.argmax(predictions.predictions, axis=-1)
        labels = predictions.label_ids
        report = classification_report(labels, preds, output_dict=True)
        ns = report['0']
        s  = report['1']
        mac = report['macro avg']['f1-score']
        print(f"{train_var:8} {test_var:8} "
              f"{ns['precision']:.3f}  {ns['recall']:.3f}  {ns['f1-score']:.3f}  "
              f"{s['precision']:.3f}  {s['recall']:.3f}  {s['f1-score']:.3f}  "
              f"{mac:.4f}")

In [ ]:
!pip install lime -q
print("LIME installed.")

In [ ]:
from lime.lime_text import LimeTextExplainer
import torch

explainer = LimeTextExplainer(class_names=['Not Sarcastic', 'Sarcastic'])

# We use the best model — en-AU trained for sarcasm
def predict_proba(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = trainer_au_sarc.model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    return probs.numpy()

print("LIME predictor ready.")

In [ ]:
# some real sarcastic examples from en-AU test set
au_test_df = test_au.to_pandas()

# examples where model got it right on sarcasm
test_ds_au = prepare_dataset(test_au, 'Sarcasm')
predictions = trainer_au_sarc.predict(test_ds_au)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Add predictions to dataframe
au_test_df['predicted'] = preds
au_test_df['actual'] = labels

# Find correctly predicted sarcastic examples
correct_sarcastic = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 1)
].head(3)

# Find wrongly predicted examples
wrong_predictions = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 0)
].head(3)

print("Correctly predicted sarcastic examples:")
print("-" * 50)
for i, row in correct_sarcastic.iterrows():
    print(f"{row['text']}")
    print()

print("Wrongly predicted examples (actual sarcastic, predicted not sarcastic):")
print("-" * 50)
for i, row in wrong_predictions.iterrows():
    print(f"{row['text']}")
    print()

In [ ]:
def predict_proba(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    # Move inputs to same device as model
    device = next(trainer_au_sarc.model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer_au_sarc.model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    return probs.cpu().numpy()

print("Fixed predictor ready.")

In [ ]:
au_test_df = test_au.to_pandas()
test_ds_au = prepare_dataset(test_au, 'Sarcasm')
predictions = trainer_au_sarc.predict(test_ds_au)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

au_test_df['predicted'] = preds
au_test_df['actual'] = labels

correct_sarcastic = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 1)
].head(3)

wrong_predictions = au_test_df[
    (au_test_df['actual'] == 1) &
    (au_test_df['predicted'] == 0)
].head(3)

sample_correct = correct_sarcastic.iloc[0]['text']
sample_wrong = wrong_predictions.iloc[0]['text']

print("Analysing correct sarcastic prediction...")
print(f"Text: {sample_correct}")
print()

exp_correct = explainer.explain_instance(
    sample_correct,
    predict_proba,
    num_features=8,
    num_samples=100
)

print("Top words pushing towards SARCASTIC:")
for word, weight in exp_correct.as_list():
    direction = "SARCASTIC ▲" if weight > 0 else "NOT SARCASTIC ▼"
    print(f"  {word:<20} {weight:+.4f}  {direction}")

In [ ]:
print("Analysing WRONG prediction (actual sarcastic, predicted not sarcastic)...")
print(f"Text: {sample_wrong}")
print()

exp_wrong = explainer.explain_instance(
    sample_wrong,
    predict_proba,
    num_features=8,
    num_samples=100
)

print("Top words pushing towards SARCASTIC:")
for word, weight in exp_wrong.as_list():
    direction = "SARCASTIC ▲" if weight > 0 else "NOT SARCASTIC ▼"
    print(f"  {word:<20} {weight:+.4f}  {direction}")

Correct prediction: Words like bad, promises, believe, core pushed towards sarcastic. The model correctly picked up on the negative sentiment words combined with political language.
Wrong prediction: Almost everything pushed towards NOT sarcastic — candidate, should, history, would. The model saw formal, educational, policy-style language and thought it was serious. But the last line "Clowns tend to be the only candidates" is the sarcastic punchline — the model missed it because the rest of the text sounds genuine.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, exp, title in zip(axes,
    [exp_correct, exp_wrong],
    ['Correct Sarcasm Detection\n(model got it right)',
     'Missed Sarcasm\n(model got it wrong)']):

    words  = [x[0] for x in exp.as_list()]
    weights = [x[1] for x in exp.as_list()]
    colours = ['#2196F3' if w > 0 else '#F44336' for w in weights]

    bars = ax.barh(words[::-1], weights[::-1], color=colours[::-1],
                   edgecolor='white', linewidth=0.8)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('LIME Weight\n(Blue = pushes towards Sarcastic, Red = pushes away)', fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('LIME Interpretability — en-AU Sarcasm Model\nWhich words drive the prediction?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lime_analysis.png', dpi=150, bbox_inches='tight')
plt.show()